# Dataset Preparation and Patch Extraction

In [1]:
import os
import glob
import json
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import tifffile as tiff

DATA_ROOT = Path(os.environ.get('HUBMAP_DATA_ROOT', './data/hubmap_pas_wsis'))
SPLIT_CONFIG_PATH = Path(os.environ.get('HUBMAP_SPLIT_CONFIG', './data/dataset_split.json'))
EXTRACTED_PATCHES_DIR = Path(os.environ.get('HUBMAP_PATCHES_DIR', './outputs/extracted_patches'))
YOLO_DATASET_DIR = Path(os.environ.get('HUBMAP_YOLO_DIR', './outputs/yolo_dataset'))

PATCH_SIZE = 1024
STRIDE_RATIO = 0.30
OVERLAP_RATIO = 1.0 - STRIDE_RATIO
BACKGROUND_RATIO_LIMIT = 0.10
MIN_BOX_SIDE_PX = 48
MIN_BOX_AREA_PX = 3600
RANDOM_SEED = 42

In [2]:
def build_official_ome_mask_registry(data_root: Path) -> dict:

    registry = {}
    for d in sorted(data_root.iterdir()):
        if not d.is_dir() or (not d.name.startswith('HBM') and len(d.name) != 9):
            continue
        for p in d.rglob('*.ome_mask.json'):
            van_prefix = p.name.split('_PAS')[0].split('-PAS')[0]
            registry[van_prefix] = p
    return registry


def resolve_slide_paths(slide_id: str, data_root: Path, ome_registry: dict) -> tuple:
    # Return the tiff_path, ome_mask_json_path pair
    slide_dir = data_root / slide_id
    tiff_candidates = [
        p for p in slide_dir.rglob('*.tif*') if 'mask' not in p.name.lower()
    ]
    if not tiff_candidates:
        raise FileNotFoundError(f'No OME-TIFF found for slide: {slide_id}')

    tiff_path = tiff_candidates[0]
    local_ome = list(slide_dir.rglob('*.ome_mask.json'))
    if local_ome:
        ome_path = local_ome[0]
    else:
        van_prefix = tiff_path.name.split('_PAS')[0].split('-PAS')[0]
        ome_path = ome_registry[van_prefix]
    return tiff_path, ome_path


def load_wsi_rgb(tiff_path: Path) -> np.ndarray:
    try:
        image = tiff.memmap(str(tiff_path))
    except Exception:
        image = tiff.imread(str(tiff_path))
    image = np.squeeze(image)
    while image.ndim > 3:
        image = np.squeeze(image[0])
    if image.ndim == 3:
        if image.shape[0] in (1, 3):
            image = np.transpose(image, (1, 2, 0))
        if image.shape[-1] > 3:
            image = image[:, :, :3]
    return image


def rasterize_geojson_polygons(json_path: Path, shape: tuple) -> np.ndarray:
    mask = np.zeros(shape, dtype=np.uint8)
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    features = data if isinstance(data, list) else data.get('features', [])
    for feat in features:
        coords = feat.get('geometry', {}).get('coordinates', [])
        if coords:
            poly = np.array(coords[0], dtype=np.int32)
            cv2.fillPoly(mask, [poly], 255)
    return mask

In [3]:
def is_valid_tissue_patch(patch_rgb: np.ndarray, is_test: bool = False, bg_threshold: float = 0.80) -> bool:
    gray = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2GRAY)
    num_pixels = patch_rgb.shape[0] * patch_rgb.shape[1]
    if (np.sum(gray <= 15) / num_pixels) > 0.99 or (np.sum(gray >= 240) / num_pixels) > 0.99:
        return False
    if (np.sum(np.std(patch_rgb, axis=-1) < 5) / num_pixels) > 0.95:
        return False
    if not is_test:
        if (np.sum(gray > 210) / num_pixels) > bg_threshold:
            return False
    return True


def extract_wsi_patches(
    slide_id: str,
    tiff_path: Path,
    ome_mask_path: Path,
    output_split_dir: Path,
    patch_size: int = PATCH_SIZE,
    stride_ratio: float = STRIDE_RATIO,
    is_test: bool = False,
) -> int:
    # Extract 1024x1024 RGB image patches and corresponding binary masks from a WSI
    stride = int(patch_size * stride_ratio)
    img_out_dir = output_split_dir / slide_id / 'images'
    msk_out_dir = output_split_dir / slide_id / 'masks'
    img_out_dir.mkdir(parents=True, exist_ok=True)
    msk_out_dir.mkdir(parents=True, exist_ok=True)

    image = load_wsi_rgb(tiff_path)
    h, w = image.shape[:2]
    mask = rasterize_geojson_polygons(ome_mask_path, (h, w))

    saved_count = 0
    for y in range(0, h, stride):
        for x in range(0, w, stride):
            y_end, x_end = min(y + patch_size, h), min(x + patch_size, w)
            img_patch = np.array(image[y:y_end, x:x_end])
            msk_patch = np.array(mask[y:y_end, x:x_end])

            if img_patch.shape[0] < patch_size or img_patch.shape[1] < patch_size:
                pad_h = patch_size - img_patch.shape[0]
                pad_w = patch_size - img_patch.shape[1]
                img_patch = cv2.copyMakeBorder(img_patch, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
                msk_patch = cv2.copyMakeBorder(msk_patch, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)

            if not is_valid_tissue_patch(img_patch, is_test=is_test):
                continue

            patch_name = f'{slide_id}_y{y}_x{x}.png'
            cv2.imwrite(str(img_out_dir / patch_name), cv2.cvtColor(img_patch, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(msk_out_dir / patch_name), msk_patch)
            saved_count += 1

    return saved_count

In [4]:
def create_yolo_dataset_with_background_cap(
    extracted_split_dir: Path,
    yolo_split_dir: Path,
    bg_ratio_limit: float = BACKGROUND_RATIO_LIMIT,
    min_side: int = MIN_BOX_SIDE_PX,
    min_area: int = MIN_BOX_AREA_PX,
    is_test: bool = False,
    seed: int = RANDOM_SEED,
) -> dict:
    # Convert binary masks to YOLO format labels and cap empty background patches at 10% for train/val
    img_dst = yolo_split_dir / 'images'
    lbl_dst = yolo_split_dir / 'labels'
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    positive_samples = []
    background_samples = []

    for msk_path in sorted(extracted_split_dir.rglob('masks/*.png')):
        img_path = msk_path.parent.parent / 'images' / msk_path.name
        mask = cv2.imread(str(msk_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        h, w = mask.shape[:2]
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        yolo_lines = []
        for cnt in contours:
            bx, by, bw, bh = cv2.boundingRect(cnt)
            if bw < min_side or bh < min_side or (bw * bh) < min_area:
                continue
            xc = (bx + bw / 2.0) / w
            yc = (by + bh / 2.0) / h
            nw = bw / w
            nh = bh / h
            yolo_lines.append(f'0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}')

        if yolo_lines:
            positive_samples.append((img_path, yolo_lines))
        else:
            background_samples.append((img_path, []))

    if not is_test and len(positive_samples) > 0:
        max_bg = int(len(positive_samples) * (bg_ratio_limit / (1.0 - bg_ratio_limit)))
        rng = random.Random(seed)
        if len(background_samples) > max_bg:
            background_samples = rng.sample(background_samples, max_bg)

    all_selected = positive_samples + background_samples
    for img_path, lines in all_selected:
        shutil.copy2(img_path, img_dst / img_path.name)
        lbl_path = lbl_dst / f'{img_path.stem}.txt'
        lbl_path.write_text('\n'.join(lines), encoding='utf-8')

    return {
        'positive_patches': len(positive_samples),
        'background_patches': len(background_samples),
        'total_patches': len(all_selected),
    }

In [5]:
# Verify *.ome_mask.json registry
with open(SPLIT_CONFIG_PATH, 'r', encoding='utf-8') as f:
    split_cfg = json.load(f)

ome_registry = build_official_ome_mask_registry(DATA_ROOT)
print(f'Indexed {len(ome_registry)} official *.ome_mask.json files in registry.')

split_summary = []
for split_key, label, is_test_flag in [('train_ids', 'TRAIN', False), ('val_ids', 'VAL', False), ('test_ids', 'TEST', True)]:
    slides = split_cfg[split_key]
    total_extracted = 0
    for sid in slides:
        slide_patch_dir = EXTRACTED_PATCHES_DIR / label.lower() / sid / 'images'
        if slide_patch_dir.exists():
            total_extracted += len(list(slide_patch_dir.glob('*.png')))
    split_summary.append((label, len(slides), total_extracted))
    print(f'[{label:5s}] WSIs={len(slides):2d} | Extracted 1024x1024 Patches (30% stride)={total_extracted:,}')

total_wsis = sum(x[1] for x in split_summary)
total_patches = sum(x[2] for x in split_summary)
print(f'Total Dataset Summary: {total_wsis} PAS WSIs -> {total_patches:,} extracted patches (100% patient-disjoint across splits).')

Indexed 35 official *.ome_mask.json files in registry.
[TRAIN] WSIs=21 | Extracted 1024x1024 Patches (30% stride)=32,040
[VAL  ] WSIs= 7 | Extracted 1024x1024 Patches (30% stride)=12,995
[TEST ] WSIs= 7 | Extracted 1024x1024 Patches (30% stride)=33,002
Total Dataset Summary: 35 PAS WSIs -> 78,037 extracted patches (100% patient-disjoint across splits).
